In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack


import joblib

In [2]:
df_original = pd.read_csv('cleaned_products.csv')
df_original.head()
df = df_original.copy()


In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   product_id         243 non-null    int64
 1   product_name       243 non-null    str  
 2   brand_name         243 non-null    str  
 3   category_name      243 non-null    str  
 4   sub_category_name  243 non-null    str  
 5   slug               243 non-null    str  
 6   thumb_image        243 non-null    str  
 7   qty                243 non-null    int64
 8   sku                243 non-null    str  
 9   price              243 non-null    int64
 10  offer_price        243 non-null    int64
 11  offer_start_price  243 non-null    str  
 12  offer_end_price    243 non-null    str  
 13  product_type       243 non-null    str  
 14  status             243 non-null    int64
 15  is_approved        243 non-null    int64
 16  short_description  243 non-null    str  
 17  long_description   243 non-

,product_id,qty,price,offer_price,status,is_approved
count,243.000000,243.000000,243.000000,243.00000,243.0,243.0
mean,139.106996,71.781893,605.465021,562.17284,1.0,1.0
std,78.913469,33.824379,1804.509748,1699.58948,0.0,0.0
min,5.000000,10.000000,4.000000,3.00000,1.0,1.0
25%,74.500000,48.000000,30.000000,25.50000,1.0,1.0
50%,135.000000,70.000000,69.000000,56.00000,1.0,1.0
75%,198.500000,90.000000,414.000000,379.00000,1.0,1.0
max,293.000000,160.000000,12500.000000,11800.00000,1.0,1.0


In [4]:
df.drop(columns = ['product_id', 'product_name', 'slug', 'thumb_image', 'qty', 'sku', 'offer_start_price', 'offer_end_price', 'status', 'is_approved', 'short_description', 'long_description', 'seo_title', 'seo_description', 'created_at', 'offer_price', 'product_type'])

,brand_name,category_name,sub_category_name,price,offer_price,product_type
0,Nestle,Grocery & Food,Instant Food,7,6,top
1,Nike,Fashion,T-shirt,25,20,featured
2,Selons,Fashion,Hoodie & Sweater,35,29,new_arrival
3,Samsung,Mobile Phones,Tablet,699,649,top
4,Xiaomi,Electronics,Smart Watch,39,34,featured
...,...,...,...,...,...,...
238,Disney,Toys & Collectibles,Plush Toys,28,23,featured
239,Pop Mart,Toys & Collectibles,Model Figures,22,18,featured
240,Pop Mart,Toys & Collectibles,Model Figures,24,19,top
241,Funko,Toys & Collectibles,Model Figures,18,15,featured


In [5]:
categorical_cols = [
    'brand_name', 
    'category_name',
    'sub_category_name'
]

numerical_cols = [
    'price'
]

Encode categorical data

In [6]:
encoder = OneHotEncoder()
cat_encoded = encoder.fit_transform(df[categorical_cols])

Scale numeric data

In [7]:
scaler = StandardScaler()
num_scaled = scaler.fit_transform(df[numerical_cols])


In [8]:
X = hstack([cat_encoded, num_scaled]) #Needed because OneHotEncoder makes dataset sparse

In [9]:
knn = NearestNeighbors(n_neighbors=11, metric='euclidean')
knn.fit(X)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [10]:
X = X.tocsr()

In [11]:
def recommend(product_index):
    distances, indices = knn.kneighbors(X[product_index])

    results = df.iloc[indices[0][1:]]
    return results

In [12]:
recommend(3)

,product_id,product_name,brand_name,category_name,sub_category_name,slug,thumb_image,qty,sku,price,...,offer_start_price,offer_end_price,product_type,status,is_approved,short_description,long_description,seo_title,seo_description,created_at
190,208,samsung galaxy tab s9 fe tablet,Samsung,Mobile Phones,Tablet,samsung-galaxy-tab-s9-fe-tablet,uploads/2026-03-14_1773517528_69b5bad86dd2a_Sa...,70,sam-tab-050,549,...,2026-03-08,2026-03-29,top,1,1,samsung galaxy tab s9 fe tabletfan edition tab...,samsung galaxy tab s9 fe tabletthis tablet off...,samsung galaxy tab s9 fe tablet,shop samsung galaxy tab s9 fe tablet for every...,2026-03-14 13:36:36
191,209,samsung galaxy tab a10 tablet,Samsung,Mobile Phones,Tablet,samsung-galaxy-tab-a10-tablet,uploads/2026-03-14_1773517551_69b5baefad524_Sa...,85,sam-tab-051,249,...,2026-03-08,2026-03-29,top,1,1,samsung galaxy tab a10 tabletaffordable tablet...,samsung galaxy tab a10 tabletthis tablet offer...,samsung galaxy tab a10 tablet,buy samsung galaxy tab a9 tablet for simple an...,2026-03-14 13:36:36
181,198,samsung galaxy z flip 3 smartphone,Samsung,Mobile Phones,Phone,samsung-galaxy-z-flip-3-smartphone,uploads/2026-03-15_1773586433_69b6c801ca6ff_Sa...,80,sam-mob-phn-040,699,...,2026-03-08,2026-03-29,top,1,1,samsung galaxy z flip 3 smartphonebalanced sam...,samsung galaxy z filp 3 smartphonethis smartph...,samsung galaxy z flip 3 smartphone,buy samsung galaxy z flip 3 smartphone with de...,2026-03-14 13:34:31
183,200,samsung galaxy s24 fe smartphone,Samsung,Mobile Phones,Phone,samsung-galaxy-s24-fe-smartphone,uploads/2026-03-14_1773516901_69b5b865149f7_Sa...,75,sam-mob-phn-042,649,...,2026-03-08,2026-03-29,top,1,1,samsung galaxy s24 fe smartphonefan edition sa...,samsung galaxy s24 fe smartphonethis smartphon...,samsung galaxy s24 fe smartphone,buy samsung galaxy s24 fe smartphone with reli...,2026-03-14 13:34:31
188,206,samsung galaxy tab s10 tablet,Samsung,Mobile Phones,Tablet,samsung-galaxy-tab-s10-tablet,uploads/2026-03-14_1773517492_69b5bab4ec214_Sa...,60,sam-tab-048,799,...,2026-03-08,2026-03-29,featured,1,1,samsung galaxy tab s10 tabletmodern android ta...,samsung galaxy tab s10 tabletthis tablet offer...,samsung galaxy tab s10 tablet,shop samsung galaxy tab s10 tablet with powerf...,2026-03-14 13:36:36
186,203,apple ipad mini 6 tablet,Apple,Mobile Phones,Tablet,apple-ipad-mini-6-tablet,uploads/2026-03-14_1773516979_69b5b8b377cf7_Ap...,50,app-tab-045,499,...,2026-03-08,2026-03-29,top,1,1,apple ipad mini 6 tabletcompact tablet designe...,apple ipad mini 6 tabletthis small form tablet...,apple ipad mini 6 tablet,shop apple ipad mini 6 tablet with compact des...,2026-03-14 13:36:36
189,207,samsung galaxy tab s10 plus tablet,Samsung,Mobile Phones,Tablet,samsung-galaxy-tab-s10-plus-tablet,uploads/2026-03-14_1773517516_69b5bacc53a60_Sa...,55,sam-tab-049,999,...,2026-03-08,2026-03-29,best,1,1,samsung galaxy tab s10 plus tabletlarge displa...,samsung galaxy tab s10 plus tabletthis tablet ...,samsung galaxy tab s10 plus tablet,buy samsung galaxy tab s10 plus tablet with pr...,2026-03-14 13:36:36
38,48,xiaomi pad 6 tablet,Xiaomi,Mobile Phones,Tablet,xiaomi-pad-6-tablet,uploads/2026-03-09_1773069702_69aee586a3f17_xi...,38,xia-mob-tab-007,329,...,2026-03-08,2026-03-27,top,1,1,xiaomi pad 6 tablethigh value tablet designed ...,xiaomi pad 6 tabletthis tablet combines clear ...,xiaomi pad 6 tablet,shop xiaomi pad 6 tablet with clear display an...,2026-03-08 11:48:52
192,210,samsung galaxy tab s8 ultra tablet,Samsung,Mobile Phones,Tablet,samsung-galaxy-tab-s8-ultra-tablet,uploads/2026-03-14_1773517577_69b5bb0947ede_Sa...,40,sam-tab-052,1099,...,2026-03-08,2026-03-29,best,1,1,samsung galaxy tab s8 ultra tablethigh end sam...,samsung galaxy tab s8 ultra tabletthis flagshi...,samsung galaxy tab s8 ultra tablet,shop samsung galaxy tab s8 ultra tablet with f...,2026-03-14 13:36:36
36,46,samsung galaxy tab a9 tablet,Samsung,Mobile Phones,Tablet,samsung-galaxy-tab-a9-tablet,uploads/2026-03-09_1773069486_69aee4ae

In [ ]:
joblib.dump({
    "knn": knn,
    "encoder": encoder,
    "scaler": scaler,
    "data": df
}, "recommender_system.joblib")